# 02 · Qwen2.5-VL-7B LoRA (원본 실험: EXP-028 → EXP-034, 추론 EXP-040)

최종 앙상블의 **7B 모델**을 재현합니다. 원본 코드는 `수인/EXP-001_masking_valid_seed43.ipynb` 셀 94(Dataset·Collator), 97(EXP-028), 103~105(EXP-034)와 `수인/EXP-040_exp034_test1280_T4.ipynb` 셀 4~7(test 추론)입니다.

| 단계 | 원본 | 데이터 | 학습률 | 비고 |
|---|---|---|---|---|
| 1 | EXP-028 | 정답별 200개 = 800개 (seed 43) | 2e-5 | 새 LoRA |
| 2 | EXP-034 | 나머지에서 정답별 200개 = 800개 (seed 143~146) | 1e-5 | weight decay 0.01 |
| 추론 | EXP-040 | valid 672 / test 6,714 | – | 해상도 1280×1280 픽셀, batch 2 |

- 학습 이미지: 384×384 픽셀, 4bit(nf4, bf16 연산), LoRA r=16 α=32 (**언어 모델 어텐션 q/k/v/o에만**, 비전 인코더 제외)
- loss: 정답 글자 1토큰에만 적용
- 원본 valid 결과: **EXP-034 단독 640/672 (95.24%)**, EXP-007과 0.52:0.48 앙상블 **646/672 (96.13%)**
- **원본과 다른 점 한 가지**: 원본 단계 2의 DataLoader는 shuffle 시드를 지정하지 않았습니다. 여기서는 재현성을 위해 seed 43을 지정했으므로 결과가 원본과 소수 문항 다를 수 있습니다.
- **원본 그대로 유지한 점**: 학습 프롬프트(`(a) ...` 형식)와 추론 프롬프트(`Question: ... a. ...` 형식)가 서로 다릅니다. 원본 EXP-040 추론이 이 방식이었으므로 그대로 둡니다.
- 권장 GPU: A100

In [ ]:
# ===== 0. 패키지 설치 (실행 후 '런타임 → 세션 다시 시작') =====
!pip install -q -U transformers accelerate peft bitsandbytes
!pip uninstall -y -q torchao          # 구버전 torchao가 peft와 충돌
!apt-get install -y -qq p7zip-full > /dev/null

In [ ]:
# ===== 1. 드라이브 연결 · 데이터 준비 =====
import os, glob, shutil, subprocess
from google.colab import drive
drive.mount("/content/drive")

SHARE_DIR = "/content/drive/MyDrive/ssafy/ai-challenge"   # 공유 폴더
OUT_DIR   = f"{SHARE_DIR}/repro"                           # 재현 결과 저장 위치 (원본 실험 파일을 덮어쓰지 않음)
os.makedirs(f"{OUT_DIR}/models", exist_ok=True)
DATA_ROOT = "/content/data"

cnt = lambda d: len(glob.glob(f"{DATA_ROOT}/{d}/*.jpg"))
if cnt("train") < 6714 or cnt("test") < 6714:
    # 드라이브에서 바로 풀면 실패할 수 있어 로컬로 복사 후 해제
    ZIP_SRC = glob.glob(f"{SHARE_DIR}/ssafy-16-2-ai*.zip")[0]
    shutil.copy(ZIP_SRC, "/content/data.zip")
    shutil.rmtree(DATA_ROOT, ignore_errors=True); os.makedirs(DATA_ROOT)
    subprocess.run(["7z","x","-y","-pocrisnotenough",f"-o{DATA_ROOT}","/content/data.zip"], check=True, stdout=subprocess.DEVNULL)
print("이미지 수 → train:", cnt("train"), "dev:", cnt("dev"), "test:", cnt("test"))
os.chdir(DATA_ROOT)   # CSV의 path 열(train/xxx.jpg)이 이 위치 기준

In [ ]:
# ===== 2. 설정 · Dataset · Collator (원본 셀 94) =====
import math, random, gc
import numpy as np, pandas as pd, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any
from tqdm.auto import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

Image.MAX_IMAGE_PIXELS = None
SEED = 43
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_ID_7B = "Qwen/Qwen2.5-VL-7B-Instruct"

train_df_7b = pd.read_csv(f"{SHARE_DIR}/split/train_split_seed43.csv").reset_index(drop=True)
valid_df = pd.read_csv(f"{SHARE_DIR}/split/valid_split_seed43.csv").reset_index(drop=True)
test_df  = pd.read_csv(f"{DATA_ROOT}/test.csv").reset_index(drop=True)

processor_train_7b = AutoProcessor.from_pretrained(MODEL_ID_7B, min_pixels=384*384, max_pixels=384*384, trust_remote_code=True)
processor_train_7b.tokenizer.padding_side = "right"

SYSTEM_INSTRUCT_TRAIN_7B = ("You are a helpful visual question answering assistant. "
                            "Answer using exactly one letter among a, b, c, or d. No explanation.")
def build_mc_prompt_train_7b(question, a, b, c, d):
    return (f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

class VQATrainDataset7B(Dataset):
    def __init__(self, dataframe): self.dataframe = dataframe.reset_index(drop=True)
    def __len__(self): return len(self.dataframe)
    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        with Image.open(row["path"]) as src: image = src.convert("RGB").copy()
        gold = str(row["answer"]).strip().lower()
        messages = [{"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT_TRAIN_7B}]},
                    {"role":"user","content":[{"type":"image","image":image},
                     {"type":"text","text":build_mc_prompt_train_7b(str(row["question"]),str(row["a"]),str(row["b"]),str(row["c"]),str(row["d"]))}]},
                    {"role":"assistant","content":[{"type":"text","text":gold}]}]
        return {"messages": messages, "image": image, "gold": gold}

@dataclass
class AnswerOnlyCollator7B:
    processor: Any
    def __call__(self, batch):
        texts = [self.processor.apply_chat_template(s["messages"], tokenize=False, add_generation_prompt=False) for s in batch]
        gold_ids = [self.processor.tokenizer.encode(s["gold"], add_special_tokens=False)[0] for s in batch]
        enc = self.processor(text=texts, images=[s["image"] for s in batch], padding=True, return_tensors="pt")
        labels = torch.full_like(enc["input_ids"], -100)
        for b, gid in enumerate(gold_ids):
            pos = (enc["input_ids"][b] == gid).nonzero(as_tuple=False).flatten()
            assert len(pos) > 0, "정답 토큰 위치 없음"
            labels[b, int(pos[-1])] = gid          # 마지막 등장 위치 = assistant 응답
        enc["labels"] = labels
        return enc

def run_training(model, df, lr, name, weight_decay=0.0, GRAD_ACCUM=4):
    loader = DataLoader(VQATrainDataset7B(df), batch_size=1, shuffle=True, collate_fn=AnswerOnlyCollator7B(processor_train_7b),
                        num_workers=0, pin_memory=False, generator=torch.Generator().manual_seed(SEED))
    lb = next(iter(loader))["labels"][0]; lt = lb[lb != -100]
    print("Loss 대상:", repr(processor_train_7b.tokenizer.decode(lt.tolist())), "| 토큰 수", len(lt)); assert len(lt) == 1
    model.train(); model.config.use_cache = False
    params = [p for p in model.parameters() if p.requires_grad]
    assert sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and "visual" in n) == 0, "비전 파라미터가 학습 대상"
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    total = math.ceil(len(loader)/GRAD_ACCUM)
    sched = get_linear_schedule_with_warmup(opt, max(1, int(total*0.03)), total)
    dev = next(model.parameters()).device
    opt.zero_grad(set_to_none=True); run = 0.0; n = 0; upd = 0
    bar = tqdm(loader, desc=name)
    for step, batch in enumerate(bar, start=1):
        batch = {k: v.to(dev) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            raw = model(**batch).loss; loss = raw / GRAD_ACCUM
        assert torch.isfinite(raw), f"비정상 loss {raw.item()}"
        loss.backward(); run += raw.item(); n += 1
        if step % GRAD_ACCUM == 0 or step == len(loader):
            torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True); upd += 1
            bar.set_postfix(loss=f"{run/n:.4f}", lr=f"{sched.get_last_lr()[0]:.2e}"); run = 0.0; n = 0
    save = f"{OUT_DIR}/models/{name}"
    model.save_pretrained(save); processor_train_7b.save_pretrained(save)
    print(f"{name} 완료 | 업데이트 {upd} | 저장 {save}")

In [ ]:
# ===== 3. 단계 1: EXP-028 (정답별 200개 = 800개, lr 2e-5) · 원본 셀 97 =====
pilot_df_7b = (pd.concat([train_df_7b[train_df_7b["answer"] == a].sample(n=200, random_state=SEED) for a in "abcd"], ignore_index=True)
               .sample(frac=1, random_state=SEED).reset_index(drop=True))

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID_7B, quantization_config=bnb, device_map="auto",
                                                          torch_dtype=torch.bfloat16, trust_remote_code=True)
base.config.use_cache = False
base = prepare_model_for_kbit_training(base)
base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model_7b = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                                           target_modules=["q_proj","k_proj","v_proj","o_proj"]))   # 비전은 qkv/proj 이름이라 제외됨
model_7b.print_trainable_parameters()
run_training(model_7b, pilot_df_7b, lr=2e-5, name="EXP-028_7b_800")

In [ ]:
# ===== 4. 단계 2: EXP-034 (새 800개, lr 1e-5, wd 0.01) · 원본 셀 103~104 =====
remaining = train_df_7b[~train_df_7b["id"].astype(str).isin(set(pilot_df_7b["id"].astype(str)))].reset_index(drop=True)
stage_df = (pd.concat([remaining[remaining["answer"].astype(str) == c].sample(n=200, random_state=SEED+100+i) for i, c in enumerate("abcd")],
                      ignore_index=True).sample(frac=1, random_state=SEED+100).reset_index(drop=True))
assert not set(stage_df["id"]) & set(pilot_df_7b["id"])
model_7b.gradient_checkpointing_enable()
run_training(model_7b, stage_df, lr=1e-5, name="EXP-034_7b_1600", weight_decay=0.01)

In [ ]:
# ===== 5. 1280 해상도 추론 (원본 EXP-040 셀 6~7) =====
model_7b.eval(); model_7b.config.use_cache = True
if hasattr(model_7b, "gradient_checkpointing_disable"): model_7b.gradient_checkpointing_disable()
gc.collect(); torch.cuda.empty_cache()
CHOICES = ["a","b","c","d"]
processor_1280 = AutoProcessor.from_pretrained(MODEL_ID_7B, min_pixels=1280*1280, max_pixels=1280*1280, trust_remote_code=True)
processor_1280.tokenizer.padding_side = "left"
choice_ids = [processor_1280.tokenizer.encode(c, add_special_tokens=False)[0] for c in CHOICES]

SYSTEM_INSTRUCT_040 = ("You are a visual question answering assistant. Carefully inspect the image and answer the "
                       "multiple-choice question. Respond with only one lowercase letter: a, b, c, or d.")
def build_mc_prompt_040(q, a, b, c, d):
    return (f"Question: {q}\n\nChoices:\na. {a}\nb. {b}\nc. {c}\nd. {d}\n\n"
            f"Answer with only one lowercase letter: a, b, c, or d.")

def predict(df, desc, batch_size=2):
    dev = next(model_7b.parameters()).device; probs = []
    for s in tqdm(range(0, len(df), batch_size), desc=desc):
        texts, imgs = [], []
        for _, row in df.iloc[s:s+batch_size].iterrows():
            with Image.open(row["path"]) as src: image = src.convert("RGB").copy()
            msgs = [{"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT_040}]},
                    {"role":"user","content":[{"type":"image","image":image},
                     {"type":"text","text":build_mc_prompt_040(str(row["question"]),str(row["a"]),str(row["b"]),str(row["c"]),str(row["d"]))}]}]
            texts.append(processor_1280.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)); imgs.append(image)
        inp = {k: v.to(dev) for k, v in processor_1280(text=texts, images=imgs, padding=True, return_tensors="pt").items()}
        with torch.inference_mode(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model_7b(**inp).logits[:, -1, :][:, choice_ids]
            p = torch.softmax(logits.float(), dim=-1)
        assert torch.isfinite(p).all(), f"비정상 확률 {s}"
        probs += p.cpu().tolist(); del inp
        if s % 100 == 0: gc.collect(); torch.cuda.empty_cache()
    return np.array(probs)

pv = predict(valid_df, "7B valid 1280")
vres = valid_df.copy(); vres["pred"] = [CHOICES[i] for i in pv.argmax(1)]; vres["correct"] = vres["pred"] == vres["answer"]
for i, c in enumerate(CHOICES): vres[f"prob_{c}"] = pv[:, i]
vres.to_csv(f"{OUT_DIR}/7b_valid_probs.csv", index=False, encoding="utf-8-sig")
print(f"7B valid: {vres.correct.mean():.4f} ({vres.correct.sum()}/672)")

pt = predict(test_df, "7B test 1280")
tres = test_df[["id"]].copy()
for i, c in enumerate(CHOICES): tres[f"prob_{c}"] = pt[:, i]
tres.to_csv(f"{OUT_DIR}/7b_test_probs.csv", index=False, encoding="utf-8-sig")
print("저장:", f"{OUT_DIR}/7b_test_probs.csv")